# Logistic Regression - Sweeps

## Setup and Imports

In [1]:
import sys

sys.path.append("..")

import wandb
import dotenv

from src.api.run import sweep_logistic_regression_ensemble
from src.api.sweep import wandb_sweep

D:\git\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
D:\git\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statemen

In [2]:
dotenv.load_dotenv()
wandb.login()

wandb: Currently logged in as: schurtenberger-david (david-schurtenberger) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Experiments

In [3]:
max_runs = 100
sweep_config = {
    "name": "Logistic Regression Ensemble",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "logistic_regression_config": {
            "parameters": {
                "penalty": {"values": ["l1", "l2", "elasticnet", None]},
                "C": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e2},
                "solver": {"values": ["lbfgs", "liblinear", "saga"]},
                "tol": {"min": 1e-5, "max": 1e-3},
                "max_iter": {"value": 1000},
                "l1_ratio": {"distribution": "uniform", "min": 0.0, "max": 1.0},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2025},
                "start_season": {"value": 2003},
                "num_features": {"distribution": "int_uniform", "min": 4, "max": 100},
                "data_loader": {"value": "season_average_ensemble"},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_logistic_regression_ensemble, run_count=max_runs, project="logistic-regression")

## Submission from Best Model

In [3]:
from src.dataloaders.ensemble import EnsembleSeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.models.logistic_regression import EnsembleLogisticRegressorModel, LogisticRegressionHyperparamConfig
from src.experiments.config import RunConfig
from src.submissions import generate_matchups, create_submission

In [4]:
run = wandb.Api().run("aicomp-mmlm/logistic-regression/ravsfw3l")
config = run.config
config

{'run_config': {'data_loader': 'season_average_ensemble',
  'num_features': 4,
  'start_season': 2003,
  'valid_season': 2025},
 'logistic_regression_config': {'C': 0.006053246988204521,
  'tol': 0.0007261930127737558,
  'solver': 'saga',
  'penalty': None,
  'l1_ratio': 0.3277738753555005,
  'max_iter': 1000}}

In [5]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = EnsembleSeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=4, valid_season=2025, start_season=2003, data_loader='season_average_ensemble', data_loader_config=None)

In [6]:
hyperparameters = LogisticRegressionHyperparamConfig(**config.get("logistic_regression_config", {}))
hyperparameters

LogisticRegressionHyperparamConfig(penalty=None, dual=False, tol=0.0007261930127737558, C=0.006053246988204521, fit_intercept=True, intercept_scaling=1.0, class_weight=None, random_state=42, solver='saga', max_iter=1000, verbose=0, warm_start=False, n_jobs=-1, l1_ratio=0.3277738753555005)

In [7]:
model = EnsembleLogisticRegressorModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [8]:
season = 2025
create_submission(
    season=season, model=model, filename=f"submission_logistic_regression_ensemble_{season}.csv", fit=True
)

D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=None)
  warnings.warn(
D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1232: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


metrics: {'train_brier_ensemble': np.float64(0.16643247178193268)}, step: 2003
metrics: {'valid_brier_ensemble': np.float64(0.17923732639338297)}, step: 2003


D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=None)
  warnings.warn(
D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1232: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


metrics: {'train_brier_ensemble': np.float64(0.16651196209704083)}, step: 2004
metrics: {'valid_brier_ensemble': np.float64(0.17670311901673602)}, step: 2004


D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=None)
  warnings.warn(
D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1232: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


...

D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=None)
  warnings.warn(
D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1232: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


metrics: {'train_brier_ensemble': np.float64(0.16533181725280363)}, step: 2023
metrics: {'valid_brier_ensemble': np.float64(0.19039361282429887)}, step: 2023


D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=None)
  warnings.warn(
D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1232: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


metrics: {'train_brier_ensemble': np.float64(0.1672553221412546)}, step: 2024
metrics: {'valid_brier_ensemble': np.float64(0.15962957077399706)}, step: 2024
metrics: {'train_brier': np.float64(0.16680181525885715)}, step: None
metrics: {'valid_brier': np.float64(0.16755124354757928)}, step: None


WindowsPath('D:/git/code/submissions/submission_logistic_regression_ensemble_2025.csv')

## Sweep with Default Features

In [3]:
max_runs = 100
sweep_config = {
    "name": "Logistic Regression Ensemble (Default Features)",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "logistic_regression_config": {
            "parameters": {
                "penalty": {"values": ["l1", "l2", "elasticnet", None]},
                "C": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e2},
                "solver": {"values": ["lbfgs", "liblinear", "saga"]},
                "tol": {"min": 1e-5, "max": 1e-3},
                "max_iter": {"value": 1000},
                "l1_ratio": {"distribution": "uniform", "min": 0.0, "max": 1.0},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2025},
                "start_season": {"value": 2003},
                "num_features": {"value": 0},
                "data_loader": {"value": "season_average_ensemble"},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_logistic_regression_ensemble, run_count=max_runs, project="logistic-regression")

## Submission from Best Model

In [9]:
from src.dataloaders.ensemble import EnsembleSeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.models.logistic_regression import EnsembleLogisticRegressorModel, LogisticRegressionHyperparamConfig
from src.experiments.config import RunConfig
from src.submissions import generate_matchups, create_submission

In [10]:
run = wandb.Api().run("aicomp-mmlm/logistic-regression/7nz78fxi")
config = run.config
config

{'run_config': {'data_loader': 'season_average_ensemble',
  'num_features': 0,
  'start_season': 2003,
  'valid_season': 2025},
 'logistic_regression_config': {'C': 0.0003038099299686865,
  'tol': 0.0007759579583211184,
  'solver': 'liblinear',
  'penalty': 'l2',
  'l1_ratio': 0.04494370809841064,
  'max_iter': 1000}}

In [11]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = EnsembleSeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=0, valid_season=2025, start_season=2003, data_loader='season_average_ensemble', data_loader_config=None)

In [12]:
hyperparameters = LogisticRegressionHyperparamConfig(**config.get("logistic_regression_config", {}))
hyperparameters

LogisticRegressionHyperparamConfig(penalty='l2', dual=False, tol=0.0007759579583211184, C=0.0003038099299686865, fit_intercept=True, intercept_scaling=1.0, class_weight=None, random_state=42, solver='liblinear', max_iter=1000, verbose=0, warm_start=False, n_jobs=-1, l1_ratio=0.04494370809841064)

In [13]:
model = EnsembleLogisticRegressorModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [14]:
season = 2025
create_submission(
    season=season,
    model=model,
    filename=f"submission_logistic_regression_ensemble_default_features_{season}.csv",
    fit=True,
)

D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 12.
  warnings.warn(


metrics: {'train_brier_ensemble': np.float64(0.16544582148574077)}, step: 2003
metrics: {'valid_brier_ensemble': np.float64(0.18104430796674786)}, step: 2003


D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 12.
  warnings.warn(


metrics: {'train_brier_ensemble': np.float64(0.16543654775116146)}, step: 2004
metrics: {'valid_brier_ensemble': np.float64(0.17910397411699558)}, step: 2004


D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 12.
  warnings.warn(


...

D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 12.
  warnings.warn(


metrics: {'train_brier_ensemble': np.float64(0.16419261192570692)}, step: 2023
metrics: {'valid_brier_ensemble': np.float64(0.19217977490979712)}, step: 2023


D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
D:\git\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 12.
  warnings.warn(


metrics: {'train_brier_ensemble': np.float64(0.1661240039708625)}, step: 2024
metrics: {'valid_brier_ensemble': np.float64(0.16211751412645403)}, step: 2024
metrics: {'train_brier': np.float64(0.16583306530383024)}, step: None
metrics: {'valid_brier': np.float64(0.16737142138812292)}, step: None


WindowsPath('D:/git/code/submissions/submission_logistic_regression_ensemble_default_features_2025.csv')